# NFL player trajectory lab
## 00 · Project readiness

**Question:** Can we validate the data contract, score trajectories correctly, and recover completed work?

The experiment has progressed beyond motion baselines: the published landing-aware residual ridge reaches **0.9269 coordinate RMSE** on 32 later validation games. This optional notebook tests the scoring contract with clearly labeled **synthetic motion**, not new NFL model training. For the research narrative, open **01 · Data analysis → 02 · Motion benchmarks**.

Predict future x/y coordinates while the pass is in the air. Whole games remain together. The 48-game holdout and official Kaggle gateway are not claimed evaluated.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from nfl_trajectory.motion import KEYS, constant_velocity, trajectory_metrics
from nfl_trajectory.report import demo, synthetic_play
from nfl_trajectory.runtime import Run

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/nfl_trajectory").is_dir())
print(f"Python {sys.version.split()[0]} | Contract verification | CPU")

### 1 · Check the scoring definition

The official metric is **coordinate RMSE**, in yards:

$$\mathrm{RMSE}=\sqrt{\frac{1}{2N}\sum_i[(x_i-\hat{x}_i)^2+(y_i-\hat{y}_i)^2]}.$$

ADE measures typical Euclidean distance; FDE measures Euclidean distance at each player's last forecast frame. We report both frame and trajectory weighting for ADE, because longer forecasts otherwise contribute more rows. A 3-yard x error and 4-yard y error gives **RMSE = √12.5**, while Euclidean distance is **5 yards**.

In [ ]:
observed, actual = synthetic_play()
predicted = constant_velocity(observed, actual[KEYS])
metrics = trajectory_metrics(actual, predicted)
display(pd.DataFrame([{"metric": key, "yards": value} for key, value in metrics.items()]).round(4))
print("Data: synthetic demonstration only. These are not NFL validation results.")

### 2 · Inspect the trajectories

The constant-velocity reference uses the last two observed frames. Its future path stays straight while these synthetic players turn. The gap is an example of the behavior a learned model must improve.

Output frame **1** means **0.1 seconds after the final observed frame**, even if the last input frame has a much larger number. We do not subtract the last input frame number from the output clock.

In [ ]:
with Run(ROOT, "notebook_demo") as run:
    demo(ROOT, run)
from IPython.display import SVG

svg_path = ROOT / "artifacts/demo/trajectory.svg"
if svg_path.exists():
    display(SVG(filename=str(svg_path)))
print("Interactive offline report: artifacts/demo/trajectory.html")

### 3 · Inspect actual NFL readiness

This section becomes populated after `nfl download` and `nfl audit`. The audit reads one weekly pair at a time, validates unique player/frame keys and forecast horizons, then writes a proposed temporal split. The actual data remains outside Git.

In [ ]:
audit_path = ROOT / "artifacts/audit_summary.json"
if audit_path.exists():
    audit = json.loads(audit_path.read_text())
    display(Markdown(f"**Actual data audit: {audit['status']}**"))
    display(pd.DataFrame(audit["pairs"]).drop(columns="games"))
    display(pd.read_csv(ROOT / "artifacts/game_splits.csv").groupby("split").agg(games=("game_id", "count"), first_date=("game_date", "min"), last_date=("game_date", "max")))
else:
    display(Markdown("**Local raw-data audit unavailable in this checkout.** The published real-data experiment is shown in notebooks 01 and 02; its presence does not recreate a local audit."))

### 4 · Know what can be used at prediction time

| Information | Use |
| --- | --- |
| Pre-throw player positions, motion, and roles | Inputs |
| Ball landing coordinates and forecast horizon | Supplied by the organizer; allowed competition inputs |
| Actual post-throw player positions | Targets only |
| Post-play outcome or statistics | Excluded from features unless availability is established |
| Future games | Validation/holdout only; never used to fit earlier-game features |

The landing location is supplied by this benchmark. A deployment that does not know it would require a different model and separate evaluation.

### 5 · Keep the evidence boundary clear

The baseline and the 2,843-candidate feature experiment have completed. **Do not repeat training merely to refresh these notebooks.** Notebook 02 compares the three feature ablations and explains the next model decision. Submission export is opt-in there; you generate and download the inference notebook from your saved weights yourself.

Retain source hashes, the dependency lock, the frozen split, checksummed model artifacts, metrics, plots, logs, and the private S3 snapshot. Completed stages are reusable; an interrupted active operation restarts. A future temporal neural model must save optimizer, scheduler, random-state, and data-order state before it can claim optimizer-level resume.